# Overview

In this module, we will inspect and explore the Training set part of the data (70% of samples). You can follow along and perform the same manipulations on the Testing set part of the data (30%) of samples.

Here are some of the topics covered in this part:
- matrix manipulations with Pandas
- exploring counts data
    - bar plots
    - heat maps
    - hierarchichal clustering
- pre-processing steps
    - log2 normalization (already performed)
    - scaling of data
- dimensional reduction
    - Principal Component Analysis

# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score
)
from sklearn import preprocessing

# Import Data

In [15]:
train_raw = pd.read_csv("train_data.csv", index_col=0)
labels_train = pd.read_csv("train_labels.csv", index_col=0)

## We can use Panda's .shape() and .head() to explore the dataframe

In [13]:
print(f"Shape of labels: {labels_train.shape}")
labels_train.head()

Shape of labels: (560, 1)


,Class
sample_154,LUAD
sample_770,BRCA
sample_738,LUAD
sample_627,BRCA
sample_258,PRAD


In [16]:
print(f"Shape of counts: {train_raw.shape}")
train_raw.head()

Shape of counts: (560, 20531)


,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
sample_154,0.0,1.423473,1.675560,2.877256,3.344910,0.0,2.765495,0.000000,0.0,0.0,...,2.776799,3.210294,3.413880,2.728206,3.510842,3.799634,3.503569,3.347155,2.668304,0.000000
sample_770,0.0,1.890250,2.193641,2.825871,3.400213,0.0,3.029003,0.553041,0.0,0.0,...,2.860263,3.322663,3.445887,2.888519,3.455706,3.706808,3.446040,3.437727,2.498032,0.000000
sample_738,0.0,1.893880,1.680050,2.823349,3.448295,0.0,2.955962,0.000000,0.0,0.0,...,2.763678,3.346187,3.458507,2.996627,3.411364,3.717604,3.514935,3.590885,3.126608,0.000000
sample_627,0.0,1.866452,2.068357,2.909781,3.424881,0.0,3.027127,0.850713,0.0,0.0,...,2.748923,3.295665,3.466790,3.122545,3.492939,3.579400,3.421555,3.461910,2.661356,0.000000
sample_258,0.0,2.050915,2.068248,2.899525,3.442621,0.0,2.910203,0.000000,0.0,0.0,...,2.871502,3.293274,3.454258,2.694288,3.404148,3.649715,3.483464,3.415907,2.851708,0.551294


## Add the tumor Class to the counts data by merging the dataframes

In [17]:
train_labeled = pd.concat([train_raw, labels_train], axis=1)

In [19]:
#note that the "Class" column is at the very end, and encodes what Class of tumor the given sample is
train_labeled.head()

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,Class
sample_154,0.0,1.423473,1.675560,2.877256,3.344910,0.0,2.765495,0.000000,0.0,0.0,...,3.210294,3.413880,2.728206,3.510842,3.799634,3.503569,3.347155,2.668304,0.000000,LUAD
sample_770,0.0,1.890250,2.193641,2.825871,3.400213,0.0,3.029003,0.553041,0.0,0.0,...,3.322663,3.445887,2.888519,3.455706,3.706808,3.446040,3.437727,2.498032,0.000000,BRCA
sample_738,0.0,1.893880,1.680050,2.823349,3.448295,0.0,2.955962,0.000000,0.0,0.0,...,3.346187,3.458507,2.996627,3.411364,3.717604,3.514935,3.590885,3.126608,0.000000,LUAD
sample_627,0.0,1.866452,2.068357,2.909781,3.424881,0.0,3.027127,0.850713,0.0,0.0,...,3.295665,3.466790,3.122545,3.492939,3.579400,3.421555,3.461910,2.661356,0.000000,BRCA
sample_258,0.0,2.050915,2.068248,2.899525,3.442621,0.0,2.910203,0.000000,0.0,0.0,...,3.293274,3.454258,2.694288,3.404148,3.649715,3.483464,3.415907,2.851708,0.551294,PRAD


In [23]:
# Now is a good time to explore the datatype of each of these columns.
print("Data type of each column:")
print(train_labeled.dtypes)

print("\nHow many columns of each datatype:")
print(train_labeled.dtypes.value_counts())

Data type of each column:
gene_0        float64
gene_1        float64
gene_2        float64
gene_3        float64
gene_4        float64
               ...   
gene_20527    float64
gene_20528    float64
gene_20529    float64
gene_20530    float64
Class          object
Length: 20532, dtype: object

How many columns of each datatype:
float64    20531
object         1
Name: count, dtype: int64
